In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
.inner_cell{font-size:20pt;}
div.text_cell_render pre code {font-size:20pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

# 1. 데이터 셋

In [4]:
import pandas as pd
url = 'https://raw.githubusercontent.com/4aix/data/refs/heads/master/ch13_apt_fillna_median.csv'
df = pd.read_csv(url)
df.sample()

,지역명,평당분양가격,연도,월
683,부산,11711.7,2017,5


- 지역명2 : 지역명필드를 라벨인코딩하여 추가(df)
- 독립변수(X) : 지역명2, 연도, 월
- 종속변수(y) : 평당분양가격
- 독립변수와 종속변수(reshape)의 스케일 조정

    * 정규화(MinMaxScaler) 작업후 : 지역명2m, 연도m, 월m 컬럼, 평당분양가격m
    * 표준화(StandardScaler) 작업후 : 지역명2s, 연도s, 월s 컬럼, 평당분양가격s
    
    => 지역명, 연도, 월, 지역명2, 지역명2m, 연도m, 월m, 평당분양가격m, 지역명2s, 연도s, 월s 컬럼, 평당분양가격s
- 데이터프레임.to_numpy

# 2. 지역명의 라벨인코딩

In [6]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['지역명2'] = le.fit_transform(df['지역명'])
df.head()

,지역명,평당분양가격,연도,월,지역명2
0,서울,18189.0,2013,12,8
1,부산,8111.0,2013,12,7
2,대구,8080.0,2013,12,5
3,인천,10204.0,2013,12,11
4,광주,6098.0,2013,12,4


# 3. MinMaxScaling

In [7]:
import numpy as np
x = df[['지역명2','연도','월']]
y = df[['평당분양가격']]
x.shape, y.shape

((2176, 3), (2176, 1))

In [8]:
from sklearn.preprocessing import MinMaxScaler
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler() 
df[['지역명2m', '연도m', '월m']] = scaler_x.fit_transform(x)
df[['평당분양가격m']] = scaler_y.fit_transform(y)
df.head()

,지역명,평당분양가격,연도,월,지역명2,지역명2m,연도m,평당분양가격m
0,서울,18189.0,2013,1.0,8,0.5000,0.0,0.328198
1,부산,8111.0,2013,1.0,7,0.4375,0.0,0.065274
2,대구,8080.0,2013,1.0,5,0.3125,0.0,0.064466
3,인천,10204.0,2013,1.0,11,0.6875,0.0,0.119878
4,광주,6098.0,2013,1.0,4,0.2500,0.0,0.012757


# 4. StandardScaling

In [13]:
from sklearn.preprocessing import StandardScaler
scaler_x = StandardScaler()
scaler_y = StandardScaler()
df[['지역명2s', '연도s', '월s']] = scaler_x.fit_transform(x)
df[['평당분양가격s']] = scaler_y.fit_transform(y)
df.head()

,지역명,평당분양가격,연도,월,지역명2,지역명2m,연도m,평당분양가격m,지역명2s,연도s,월s,평당분양가격s
0,서울,18189.0,2013,1.0,8,0.5000,0.0,0.328198,0.000000,-1.875367,1.62196,1.168591
1,부산,8111.0,2013,1.0,7,0.4375,0.0,0.065274,-0.204124,-1.875367,1.62196,-0.728312
2,대구,8080.0,2013,1.0,5,0.3125,0.0,0.064466,-0.612372,-1.875367,1.62196,-0.734147
3,인천,10204.0,2013,1.0,11,0.6875,0.0,0.119878,0.612372,-1.875367,1.62196,-0.334363
4,광주,6098.0,2013,1.0,4,0.2500,0.0,0.012757,-0.816497,-1.875367,1.62196,-1.107203


# 5. 지역명을 원핫인코딩


In [14]:
df.loc[:16, ['지역명2', '지역명']].sort_values(by='지역명2')

,지역명2,지역명
9,0,강원
7,1,경기
15,2,경남
14,3,경북
4,4,광주
2,5,대구
5,6,대전
1,7,부산
0,8,서울
8,9,세종


In [15]:
loc = list(df.loc[:16].sort_values(by='지역명2')['지역명'])
print('원핫인코딩 후 열이름 :', loc)

원핫인코딩 후 열이름 : ['강원', '경기', '경남', '경북', '광주', '대구', '대전', '부산', '서울', '세종', '울산', '인천', '전남', '전북', '제주', '충남', '충북']


In [16]:
from tensorflow.keras.utils import to_categorical
temp1 = to_categorical(df['지역명2'])
temp2 = pd.get_dummies(df['지역명2']).values
np.all(temp1==temp2)

True

In [17]:
# 원핫인코딩 방법1 : to_categorical
df[loc] = to_categorical(df['지역명2'])
df.head()

,지역명,평당분양가격,연도,월,지역명2,지역명2m,연도m,평당분양가격m,지역명2s,연도s,...,부산,서울,세종,울산,인천,전남,전북,제주,충남,충북
0,서울,18189.0,2013,1.0,8,0.5000,0.0,0.328198,0.000000,-1.875367,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,부산,8111.0,2013,1.0,7,0.4375,0.0,0.065274,-0.204124,-1.875367,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,대구,8080.0,2013,1.0,5,0.3125,0.0,0.064466,-0.612372,-1.875367,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,인천,10204.0,2013,1.0,11,0.6875,0.0,0.119878,0.612372,-1.875367,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,광주,6098.0,2013,1.0,4,0.2500,0.0,0.012757,-0.816497,-1.875367,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
df.drop(loc, axis=1, inplace=True)

In [19]:
# 원핫인코딩 방법2 : get_dummies
df[loc] = pd.get_dummies(df['지역명2'])
df.head()

,지역명,평당분양가격,연도,월,지역명2,지역명2m,연도m,평당분양가격m,지역명2s,연도s,...,부산,서울,세종,울산,인천,전남,전북,제주,충남,충북
0,서울,18189.0,2013,1.0,8,0.5000,0.0,0.328198,0.000000,-1.875367,...,0,1,0,0,0,0,0,0,0,0
1,부산,8111.0,2013,1.0,7,0.4375,0.0,0.065274,-0.204124,-1.875367,...,1,0,0,0,0,0,0,0,0,0
2,대구,8080.0,2013,1.0,5,0.3125,0.0,0.064466,-0.612372,-1.875367,...,0,0,0,0,0,0,0,0,0,0
3,인천,10204.0,2013,1.0,11,0.6875,0.0,0.119878,0.612372,-1.875367,...,0,0,0,0,1,0,0,0,0,0
4,광주,6098.0,2013,1.0,4,0.2500,0.0,0.012757,-0.816497,-1.875367,...,0,0,0,0,0,0,0,0,0,0


In [20]:
df.shape

(2176, 29)